In [ ]:
import os
import pandas as pd
import numpy as np

import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam


In [34]:
# Store Info

from pathlib import Path

seed = 0
np.random.seed(seed)
tf.random.set_seed(seed)




def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


store_cols = [
    "행정동코드",
    "상권업종중분류코드",
    "상권업종중분류명",
]

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw_data"

store_path = DATA_DIR / "store_info" / "상가(상권)정보_서울.csv"

store_df = read_csv_kor(store_path, usecols=store_cols)
# store_df = read_csv_kor(store_path)  # 전체 컬럼 읽기

# display(store_df.head())
store_df.head()


,상권업종중분류코드,상권업종중분류명,행정동코드
0,I210,기타 간이,11740580
1,N104,고용 알선,11170530
2,I212,비알코올,11500620
3,R104,유원지·오락,11350670
4,G217,시계·귀금속 소매,11110615


In [22]:
# Sales Info


def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


sales_cols = [
    "기준_년분기_코드",
    "행정동_코드",
    "서비스_업종_코드_명",
    "당월_매출_금액",
    "월요일_매출_금액",
    "화요일_매출_금액",
    "수요일_매출_금액",
    "목요일_매출_금액",
    "금요일_매출_금액",
    "토요일_매출_금액",
    "일요일_매출_금액",
    "시간대_00~06_매출_금액",
    "시간대_06~11_매출_금액",
    "시간대_11~14_매출_금액",
    "시간대_14~17_매출_금액",
    "시간대_17~21_매출_금액",
    "시간대_21~24_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액",
    "연령대_10_매출_금액",
    "연령대_20_매출_금액",
    "연령대_30_매출_금액",
    "연령대_40_매출_금액",
    "연령대_50_매출_금액",
    "연령대_60_이상_매출_금액",
]

sales_path = DATA_DIR / "sales_info" / "서울시(추정매출-행정동).csv"

sales_df = read_csv_kor(sales_path, usecols=sales_cols)
# sales_df = read_csv_kor(sales_path) # 전체 컬럼 읽기

# display(store_df.head())
sales_df.head()


,기준_년분기_코드,행정동_코드,서비스_업종_코드_명,당월_매출_금액,월요일_매출_금액,화요일_매출_금액,수요일_매출_금액,목요일_매출_금액,금요일_매출_금액,토요일_매출_금액,...,시간대_17~21_매출_금액,시간대_21~24_매출_금액,남성_매출_금액,여성_매출_금액,연령대_10_매출_금액,연령대_20_매출_금액,연령대_30_매출_금액,연령대_40_매출_금액,연령대_50_매출_금액,연령대_60_이상_매출_금액
0,20253,11740700,전자상거래업,10751618,0,1075162,3225485,1075162,2150324,3225485,...,4945744,4569438,6450971,4300647,0,0,3655550,0,4031857,3064211
1,20253,11740700,조명용품,8249940,1485258,1337594,1078706,1236916,484219,1639632,...,1049940,0,6250490,1985068,0,76708,603421,1556581,1643437,4355411
2,20253,11740700,인테리어,661900993,131389877,81770054,123617712,55580120,135020022,134523208,...,64033698,0,378222924,191416706,0,5562102,46276687,147256645,135249608,235294587
3,20253,11740700,철물점,115789484,24534077,39113866,4569884,19009881,23975851,3471315,...,36905738,0,68777997,41101758,0,227558,16804451,22900671,40435377,29511697
4,20253,11740700,가구,13984669,462456,3463794,1461362,0,0,7214314,...,7214314,0,2358525,11626144,0,0,0,0,7214314,6770355


In [ ]:
# Population Info


def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


population_cols = [
    "행정동_코드",
    "총_유동인구_수",
    "남성_유동인구_수",
    "여성_유동인구_수",
    "연령대_20_유동인구_수",
    "연령대_30_유동인구_수",
    "연령대_40_유동인구_수",
    "시간대_06_11_유동인구_수",
    "시간대_11_14_유동인구_수",
    "시간대_14_17_유동인구_수",
    "시간대_17_21_유동인구_수",
    "시간대_21_24_유동인구_수",
    "월요일_유동인구_수",
    "화요일_유동인구_수",
    "수요일_유동인구_수",
    "목요일_유동인구_수",
    "금요일_유동인구_수",
    "토요일_유동인구_수",
    "일요일_유동인구_수"
]

population_path = DATA_DIR / "population_info" / "서울시(유동인구-행정동).csv"

if not population_path.exists():
    raise FileNotFoundError(f"파일이 없습니다: {population_path}")

population_df = read_csv_kor(population_path, usecols=population_cols)
# population_df = read_csv_kor(population_path)  # 전체 컬럼 읽기

population_df.head()


In [ ]:
# 데이터 컬럼 모아보기

files = {
    "store": store_df,
    "sales": sales_df,
    "population": population_df
}

for name, df in files.items():
    print(f"\n📌 {name} 데이터 컬럼 ")
    print(df.columns)


In [ ]:
import pandas as pd

def show_unique(df, col, show_count=True, show_freq=False, top_n=20):
    """
    df : 데이터프레임
    col : 확인할 컬럼명 (문자열)
    show_count : 고유값 개수 출력 여부
    show_freq : 각 값의 빈도수 출력 여부
    top_n : 빈도 출력 시 상위 n개
    """

    if col not in df.columns:
        print(f"❌ '{col}' 컬럼이 존재하지 않습니다.")
        print("현재 컬럼 목록:", list(df.columns))
        return

    print(f"\n📌 [{col}] 컬럼 분석")

    # 공백 제거
    series = df[col].astype(str).str.strip()

    # 고유값
    unique_vals = sorted(series.dropna().unique())

    if show_count:
        print(f"✅ 고유값 개수: {len(unique_vals)}")

    print("\n🔹 고유값 목록:")
    for v in unique_vals:
        print(v)

    if show_freq:
        print(f"\n🔹 빈도 상위 {top_n}개:")
        freq = series.value_counts().head(top_n)
        print(freq)


# =========================
# 사용 예시
# =========================

# store 중분류명 고유값 확인
show_unique(store_df, "상권업종소분류명")

# store 중분류코드 확인
show_unique(store_df, "상권업종중분류명")



# sales 업종명 + 빈도까지 보고 싶을 때
# show_unique(sales, "서비스_업종_코드_명", show_freq=True)




📌 [상권업종소분류명] 컬럼 분석
✅ 고유값 개수: 247

🔹 고유값 목록:
PC방
가구 소매업
가발 소매업
가방 소매업
가스 충전소
가전제품 소매업
가전제품 수리업
가정용 연료 소매업
가죽/가방/신발 수선업
가축 사료 소매업
건강보조식품 소매업
건물 및 토목 엔지니어링 서비스업
건설/건축자재 소매업
건설기계/장비 대여업
건어물/젓갈 소매업
건축 설계 및 관련 서비스업
건축물 일반 청소업
결혼 상담 서비스업
경양식
경영 컨설팅업
고용 알선업
곡물/곡분 소매업
골프 연습장
곱창 전골/구이
공인노무사
공인회계사
광고 대행업
광고 매체 판매업
광고물 설계/제작업
교육컨설팅업
구내식당
국/탕/찌개류
국수/칼국수
그 외 기타 간이 음식점
그 외 기타 개인/가정용품 수리업
그 외 기타 교육기관
그 외 기타 상품 전문 소매업
그 외 기타 숙박업
그 외 기타 종합 소매업
기념품점
기숙사/고시원
기타 개인/가정용품 대여업
기타 건설/건축자재 소매업
기타 광고 관련 서비스업
기타 교육지원 서비스업
기타 기술/직업 훈련학원
기타 동남아식 전문
기타 법무관련 서비스업
기타 사무 지원 서비스업
기타 산업용 기계/장비 대여업
기타 서양식 음식점
기타 스포츠시설 운영업
기타 엔지니어링 서비스업
기타 여행 보조/예약 서비스업
기타 예술/스포츠 교육기관
기타 오락관련 서비스업
기타 오락장
기타 운송장비 대여업
기타 의류 소매업
기타 의원
기타 일식 음식점
기타 한식 음식점
기타 회계 관련 서비스업
김밥/만두/분식
꽃집
낚시터 운영업
남성 의류 소매업
내과/소아과 의원
냉면/밀면
네일숍
노래방
닭/오리고기 구이/찜
담배/전자담배 소매업
당구장
도시 계획 및 조경 설계 서비스업
독서실/스터디 카페
동물병원
돼지고기 구이/찜
떡/한과
레크리에이션 교육기관
마라탕/훠궈
마사지/안마
만화방
명함/간판/광고물 제작
모터사이클 및 부품 소매업
모터사이클 수리업
목욕탕/사우나
무도 유흥 주점
문구/회화용품 소매업
미술학원
미용실
바둑/장기/체스 경기 운영업
반찬/식료품 소매업
방사선 진단/병리 검사 

In [23]:
import pandas as pd

def show_unique(df, col, show_count=True, show_freq=False, top_n=20):
    """
    df : 데이터프레임
    col : 확인할 컬럼명 (문자열)
    show_count : 고유값 개수 출력 여부
    show_freq : 각 값의 빈도수 출력 여부
    top_n : 빈도 출력 시 상위 n개
    """

    if col not in df.columns:
        print(f"❌ '{col}' 컬럼이 존재하지 않습니다.")
        print("현재 컬럼 목록:", list(df.columns))
        return

    print(f"\n📌 [{col}] 컬럼 분석")

    # 공백 제거
    series = df[col].astype(str).str.strip()

    # 고유값
    unique_vals = sorted(series.dropna().unique())

    if show_count:
        print(f"✅ 고유값 개수: {len(unique_vals)}")

    print("\n🔹 고유값 목록:")
    for v in unique_vals:
        print(v)

    if show_freq:
        print(f"\n🔹 빈도 상위 {top_n}개:")
        freq = series.value_counts().head(top_n)
        print(freq)


# =========================
# 사용 예시
# =========================

# store 중분류명 고유값 확인
show_unique(sales_df, "서비스_업종_코드_명")


# sales 업종명 + 빈도까지 보고 싶을 때
# show_unique(sales, "서비스_업종_코드_명", show_freq=True)




📌 [서비스_업종_코드_명] 컬럼 분석
✅ 고유값 개수: 63

🔹 고유값 목록:
PC방
가구
가방
가전제품
가전제품수리
고시원
골프연습장
네일숍
노래방
당구장
문구
미곡판매
미용실
반찬가게
부동산중개업
분식전문점
서적
섬유제품
세탁소
수산물판매
슈퍼마켓
스포츠 강습
스포츠클럽
시계및귀금속
신발
안경
애완동물
양식음식점
여관
예술학원
완구
외국어학원
운동/경기용품
육류판매
의료기기
의약품
인테리어
일반교습학원
일반의류
일반의원
일식음식점
자동차미용
자동차수리
자전거 및 기타운송장비
전자상거래업
제과점
조명용품
중식음식점
철물점
청과상
치과의원
치킨전문점
커피-음료
컴퓨터및주변장치판매
패스트푸드점
편의점
피부관리실
한식음식점
한의원
핸드폰
호프-간이주점
화장품
화초


In [35]:
import pandas as pd
import re

# ==============================
# 🔹 설정: 확인할 컬럼명 입력
# ==============================
col = "상권업종중분류명"   # <- 필요하면 여기 컬럼명만 바꿔

if col not in store_df.columns:
    raise ValueError(f"'{col}' 컬럼이 존재하지 않습니다.\n현재 컬럼: {list(store_df.columns)}")

s_raw = store_df[col]

print("====================================")
print(f"📌 [{col}] 컬럼 업종 개수 분석 시작")
print("====================================\n")

# =================================
# 1️⃣ 원본 기준 고유값 개수
# =================================
print("1️⃣ 원본 기준 (dropna 적용)")
raw_nunique = s_raw.dropna().nunique()
print("원본 nunique:", raw_nunique)


# =================================
# 2️⃣ 문자열 정규화 함수 정의
# =================================
def normalize_text(x):
    if pd.isna(x):
        return pd.NA

    x = str(x)

    # 특수 공백 제거
    x = x.replace("\u00A0", " ")  # NBSP
    x = re.sub(r"[\u200B-\u200D\uFEFF]", "", x)  # zero-width 제거

    # 앞뒤 공백 제거
    x = x.strip()

    # 연속 공백 하나로 통일
    x = re.sub(r"\s+", " ", x)

    if x == "":
        return pd.NA

    return x


# =================================
# 3️⃣ 정규화 후 고유값 개수
# =================================
s_norm = s_raw.map(normalize_text)

norm_nunique = s_norm.dropna().nunique()

print("\n2️⃣ 정규화 후 (공백/특수문자 제거)")
print("정규화 nunique:", norm_nunique)


# =================================
# 4️⃣ 차이 원인 분석
# =================================
print("\n====================================")
print("🔎 차이 원인 분석")
print("====================================")

if raw_nunique == norm_nunique:
    print("👉 원본과 정규화 개수가 동일함.")
    print("👉 직접 센 숫자에 중복 포함 가능성 있음.")
else:
    print("👉 원본과 정규화 개수가 다름.")
    print("👉 공백/특수문자 때문에 같은 업종이 여러 개로 인식되었을 가능성 있음.")

    tmp = pd.DataFrame({"raw": s_raw, "norm": s_norm}).dropna()

    multi_raw = (
        tmp.groupby("norm")["raw"]
        .nunique()
        .sort_values(ascending=False)
    )

    suspects = multi_raw[multi_raw > 1].index.tolist()

    print("\n⚠️ 정규화하면 합쳐지는 값들:")
    print("개수:", len(suspects))

    for key in suspects[:20]:  # 상위 20개만 출력
        raws = sorted(tmp.loc[tmp["norm"] == key, "raw"].unique())
        print("\n[정규화 후]:", key)
        print(" -> 원본 값들:", raws)


# =================================
# 5️⃣ 최종 고유값 목록 출력
# =================================
print("\n====================================")
print("📋 최종 정규화 기준 고유 업종 목록")
print("====================================\n")

unique_vals = sorted(s_norm.dropna().unique())

for v in unique_vals:
    print(v)

print("\n====================================")
print("🎯 최종 정규화 기준 업종 개수:", len(unique_vals))
print("====================================")


📌 [상권업종중분류명] 컬럼 업종 개수 분석 시작

1️⃣ 원본 기준 (dropna 적용)
원본 nunique: 75

2️⃣ 정규화 후 (공백/특수문자 제거)
정규화 nunique: 75

🔎 차이 원인 분석
👉 원본과 정규화 개수가 동일함.
👉 직접 센 숫자에 중복 포함 가능성 있음.

📋 최종 정규화 기준 고유 업종 목록

가구 소매
가전·통신 소매
가전제품 수리
가정용품 대여
고용 알선
광고
교육 지원
구내식당·뷔페
기술 서비스
기타 가정용품 수리
기타 간이
기타 개인
기타 교육
기타 보건
기타 사업 서비스
기타 상품 소매
기타 생활용품 소매
기타 숙박
기타 외국
기타 전문 과학
담배 소매
도서관·사적지
동남아시아
모터사이클 소매
모터사이클 수리
법무관련
병원
본사·경영 컨설팅
부동산 서비스
비알코올
사무 지원
사진 촬영
산업용품 대여
서양식
섬유·의복·신발 소매
세탁
수의
스포츠 서비스
시계·귀금속 소매
시설관리
시장 조사
식료품 소매
식물 소매
안경·정밀기기 소매
애완동물·용품 소매
여행사·보조
연료 소매
오락용품 소매
욕탕·신체관리
운송장비 대여
유원지·오락
음료 소매
의약·화장품 소매
의원
이용·미용
인쇄·제품제작
일반 교육
일반 숙박
일식
자동차 부품 소매
자동차 수리·세차
장례식장
장식품 소매
전문 디자인
조경·유지
종합 소매
주점
중고 상품 소매
중식
철물·건설자재 소매
청소·방제
컴퓨터 수리
통신장비 수리
한식
회계·세무

🎯 최종 정규화 기준 업종 개수: 75


In [36]:
import pandas as pd
import re
from pathlib import Path

# =========================
# 0) 서비스 업종(63개) - (참고용, 검증용)
# =========================
SERVICE_CATS = [
    "PC방","가구","가방","가전제품","가전제품수리","고시원","골프연습장","네일숍","노래방","당구장",
    "문구","미곡판매","미용실","반찬가게","부동산중개업","분식전문점","서적","섬유제품","세탁소","수산물판매",
    "슈퍼마켓","스포츠 강습","스포츠클럽","시계및귀금속","신발","안경","애완동물","양식음식점","여관","예술학원",
    "완구","외국어학원","운동/경기용품","육류판매","의료기기","의약품","인테리어","일반교습학원","일반의류","일반의원",
    "일식음식점","자동차미용","자동차수리","자전거 및 기타운송장비","전자상거래업","제과점","조명용품","중식음식점","철물점","청과상",
    "치과의원","치킨전문점","커피-음료","컴퓨터및주변장치판매","패스트푸드점","편의점","피부관리실","한식음식점","한의원","핸드폰",
    "호프-간이주점","화장품","화초"
]
SERVICE_SET = set(SERVICE_CATS)

# =========================
# 1) 중분류 75개(너가 준 리스트)
# =========================
MID_CATS_75 = [
"가구 소매","가전·통신 소매","가전제품 수리","가정용품 대여","고용 알선","광고","교육 지원","구내식당·뷔페",
"기술 서비스","기타 가정용품 수리","기타 간이","기타 개인","기타 교육","기타 보건","기타 사업 서비스","기타 상품 소매",
"기타 생활용품 소매","기타 숙박","기타 외국","기타 전문 과학","담배 소매","도서관·사적지","동남아시아",
"모터사이클 소매","모터사이클 수리","법무관련","병원","본사·경영 컨설팅","부동산 서비스","비알코올","사무 지원",
"사진 촬영","산업용품 대여","서양식","섬유·의복·신발 소매","세탁","수의","스포츠 서비스","시계·귀금속 소매",
"시설관리","시장 조사","식료품 소매","식물 소매","안경·정밀기기 소매","애완동물·용품 소매","여행사·보조",
"연료 소매","오락용품 소매","욕탕·신체관리","운송장비 대여","유원지·오락","음료 소매","의약·화장품 소매","의원",
"이용·미용","인쇄·제품제작","일반 교육","일반 숙박","일식","자동차 부품 소매","자동차 수리·세차","장례식장",
"장식품 소매","전문 디자인","조경·유지","종합 소매","주점","중고 상품 소매","중식","철물·건설자재 소매",
"청소·방제","컴퓨터 수리","통신장비 수리","한식","회계·세무"
]

# =========================
# 2) 텍스트 정규화
# =========================
def norm_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\u00A0", " ")
    x = re.sub(r"[\u200B-\u200D\uFEFF]", "", x)  # zero-width 제거
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return x

# =========================
# 3) "중분류(75) -> 서비스업종(63)" 매핑 테이블(초안)
#    - 애매/불가능하면 '기타(매핑검토)'로 보내고 나중에 수동 보정
# =========================
MAP_MID_TO_SERVICE = {
    # 소매/판매
    "가구 소매": "가구",
    "가전·통신 소매": "가전제품",               # 통신 포함이지만 중분류가 뭉뚱그려져 있어 우선 가전으로
    "담배 소매": "편의점",
    "식료품 소매": "슈퍼마켓",
    "종합 소매": "슈퍼마켓",
    "섬유·의복·신발 소매": "일반의류",
    "시계·귀금속 소매": "시계및귀금속",
    "안경·정밀기기 소매": "안경",
    "애완동물·용품 소매": "애완동물",
    "식물 소매": "화초",
    "철물·건설자재 소매": "철물점",
    "오락용품 소매": "완구",
    "음료 소매": "커피-음료",
    "의약·화장품 소매": "화장품",               # 의약+화장품 묶음이라 우선 화장품(필요시 의약품으로 분기)
    "자동차 부품 소매": "자동차수리",            # 부품 판매 항목이 따로 없어 정비/수리로 흡수
    "모터사이클 소매": "자전거 및 기타운송장비",
    "장식품 소매": "시계및귀금속",
    "기타 상품 소매": "슈퍼마켓",
    "기타 생활용품 소매": "문구",                 # 생활용품 소매는 케이스별로 다르니 임시(검토 가능)

    # 수리/정비/미용
    "가전제품 수리": "가전제품수리",
    "기타 가정용품 수리": "가전제품수리",
    "자동차 수리·세차": "자동차수리",
    "컴퓨터 수리": "컴퓨터및주변장치판매",        # 수리 카테고리 부재 → 가장 가까운 컴퓨터 카테고리로
    "통신장비 수리": "핸드폰",
    "모터사이클 수리": "자전거 및 기타운송장비",
    "이용·미용": "미용실",
    "욕탕·신체관리": "피부관리실",
    "세탁": "세탁소",

    # 오락/스포츠
    "유원지·오락": "노래방",                      # 오락실/유원지 항목 부재 → 오락 대표로 임시
    "스포츠 서비스": "스포츠클럽",
    "운송장비 대여": "자전거 및 기타운송장비",

    # 숙박/부동산
    "일반 숙박": "여관",
    "기타 숙박": "여관",
    "부동산 서비스": "부동산중개업",

    # 의료
    "병원": "일반의원",
    "의원": "일반의원",
    "수의": "애완동물",                           # 동물병원/수의 → 애완동물로 흡수(검토 가능)

    # 음식/주점
    "한식": "한식음식점",
    "중식": "중식음식점",
    "일식": "일식음식점",
    "서양식": "양식음식점",
    "구내식당·뷔페": "한식음식점",
    "기타 간이": "패스트푸드점",
    "주점": "호프-간이주점",
    "비알코올": "커피-음료",

    # 교육(애매: 예술/외국어/일반 교습으로 더 세분 가능)
    "일반 교육": "일반교습학원",
    "기타 교육": "일반교습학원",
    "교육 지원": "일반교습학원",

    # 그 외: 서비스 63개에 대응이 애매하거나 B2B(매출데이터 체계에 없을 가능성 큼)
    "가정용품 대여": "기타(매핑검토)",
    "고용 알선": "기타(매핑검토)",
    "광고": "기타(매핑검토)",
    "기술 서비스": "기타(매핑검토)",
    "기타 개인": "기타(매핑검토)",
    "기타 보건": "기타(매핑검토)",
    "기타 사업 서비스": "기타(매핑검토)",
    "기타 외국": "기타(매핑검토)",
    "기타 전문 과학": "기타(매핑검토)",
    "도서관·사적지": "서적",
    "동남아시아": "기타(매핑검토)",
    "법무관련": "기타(매핑검토)",
    "본사·경영 컨설팅": "기타(매핑검토)",
    "사무 지원": "기타(매핑검토)",
    "사진 촬영": "기타(매핑검토)",
    "산업용품 대여": "기타(매핑검토)",
    "여행사·보조": "기타(매핑검토)",
    "연료 소매": "기타(매핑검토)",
    "인쇄·제품제작": "기타(매핑검토)",
    "장례식장": "기타(매핑검토)",
    "전문 디자인": "기타(매핑검토)",
    "조경·유지": "인테리어",
    "중고 상품 소매": "기타(매핑검토)",
    "청소·방제": "기타(매핑검토)",
    "시설관리": "기타(매핑검토)",
    "시장 조사": "기타(매핑검토)",
    "회계·세무": "기타(매핑검토)",
}

# =========================
# 4) 매핑 함수(정규화 후 exact 매칭)
# =========================
def map_mid_to_service(mid_name: str) -> str:
    n = norm_text(mid_name)
    if not n:
        return "기타(매핑검토)"
    return MAP_MID_TO_SERVICE.get(n, "기타(매핑검토)")

# =========================
# 5) "75개 리스트 기준"으로 매핑테이블 생성(검증용)
# =========================
mapping_df = pd.DataFrame({"상권업종중분류명": MID_CATS_75})
mapping_df["상권업종중분류명_norm"] = mapping_df["상권업종중분류명"].map(norm_text)
mapping_df["서비스업종"] = mapping_df["상권업종중분류명_norm"].map(map_mid_to_service)

# 매핑 품질 체크
mapping_df["매핑상태"] = mapping_df["서비스업종"].apply(lambda x: "OK" if x in SERVICE_SET else "검토필요")
ok_cnt = (mapping_df["매핑상태"] == "OK").sum()
review_cnt = (mapping_df["매핑상태"] == "검토필요").sum()

print(f"✅ 중분류(75) 매핑 결과: OK={ok_cnt}, 검토필요={review_cnt}")
print("\n--- 🔎 검토필요 항목 ---")
print(mapping_df.loc[mapping_df["매핑상태"]=="검토필요", ["상권업종중분류명","서비스업종"]].to_string(index=False))

# =========================
# 6) store 전체에 매핑 붙이기
# =========================
# store에 필요한 컬럼 체크
need_cols = ["상권업종중분류명"]
missing = [c for c in need_cols if c not in store.columns]
if missing:
    raise ValueError(f"store에 필요한 컬럼이 없음: {missing}\n현재 컬럼: {list(store.columns)}")

store_mapped = store.copy()
store_mapped["상권업종중분류명_norm"] = store_mapped["상권업종중분류명"].map(norm_text)
store_mapped["서비스업종"] = store_mapped["상권업종중분류명_norm"].map(map_mid_to_service)
store_mapped["매핑상태"] = store_mapped["서비스업종"].apply(lambda x: "OK" if x in SERVICE_SET else "검토필요")

print("\n✅ store 매핑 완료(샘플 10개)")
print(store_mapped[["상권업종중분류명","서비스업종","매핑상태"]].head(10).to_string(index=False))

# =========================
# 7) 저장
# =========================
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

mapping_path = out_dir / "mapping_mid75_to_service63.csv"
store_path = out_dir / "store_with_service_mapping.csv"
review_path = out_dir / "mapping_review_needed.csv"

mapping_df.to_csv(mapping_path, index=False, encoding="utf-8-sig")
store_mapped.to_csv(store_path, index=False, encoding="utf-8-sig")
mapping_df.loc[mapping_df["매핑상태"]=="검토필요"].to_csv(review_path, index=False, encoding="utf-8-sig")

print("\n💾 저장 완료")
print(" -", mapping_path.resolve())
print(" -", store_path.resolve())
print(" -", review_path.resolve())


✅ 중분류(75) 매핑 결과: OK=50, 검토필요=25

--- 🔎 검토필요 항목 ---
 상권업종중분류명    서비스업종
  가정용품 대여 기타(매핑검토)
    고용 알선 기타(매핑검토)
       광고 기타(매핑검토)
   기술 서비스 기타(매핑검토)
    기타 개인 기타(매핑검토)
    기타 보건 기타(매핑검토)
기타 사업 서비스 기타(매핑검토)
    기타 외국 기타(매핑검토)
 기타 전문 과학 기타(매핑검토)
    동남아시아 기타(매핑검토)
     법무관련 기타(매핑검토)
본사·경영 컨설팅 기타(매핑검토)
    사무 지원 기타(매핑검토)
    사진 촬영 기타(매핑검토)
  산업용품 대여 기타(매핑검토)
     시설관리 기타(매핑검토)
    시장 조사 기타(매핑검토)
   여행사·보조 기타(매핑검토)
    연료 소매 기타(매핑검토)
  인쇄·제품제작 기타(매핑검토)
     장례식장 기타(매핑검토)
   전문 디자인 기타(매핑검토)
 중고 상품 소매 기타(매핑검토)
    청소·방제 기타(매핑검토)
    회계·세무 기타(매핑검토)

✅ store 매핑 완료(샘플 10개)
   상권업종중분류명    서비스업종 매핑상태
      기타 간이   패스트푸드점   OK
      고용 알선 기타(매핑검토) 검토필요
      비알코올     커피-음료   OK
     유원지·오락      노래방   OK
  시계·귀금속 소매   시계및귀금속   OK
    부동산 서비스   부동산중개업   OK
    부동산 서비스   부동산중개업   OK
섬유·의복·신발 소매     일반의류   OK
         주점  호프-간이주점   OK
      가구 소매       가구   OK

💾 저장 완료
 - /Users/gimgyumin/Documents/commercial-area-analysis-ai/ai/outputs/mapping_mid75_to_service63.csv
 - /Users/gimgyumin/Documents/commerc

In [37]:
import pandas as pd

# 이미 store_mapped가 있다고 가정
# (상권업종중분류명, 서비스업종, 매핑상태 포함)

print("📌 전체 행 수:", len(store_mapped))

# 1️⃣ 매핑 성공한 것만 유지
store_final = store_mapped[store_mapped["매핑상태"] == "OK"].copy()

print("📌 B2B 제외 후 행 수:", len(store_final))

# 2️⃣ 사용되는 서비스 업종 개수 확인
print("📌 최종 서비스 업종 개수:",
      store_final["서비스업종"].nunique())

# 3️⃣ 최종 업종 목록 확인
print("\n📌 최종 서비스 업종 리스트:")
print(sorted(store_final["서비스업종"].unique()))

# 4️⃣ 저장
store_final.to_csv("outputs/store_final_filtered.csv",
                   index=False,
                   encoding="utf-8-sig")

print("\n✅ 저장 완료: outputs/store_final_filtered.csv")


📌 전체 행 수: 534978
📌 B2B 제외 후 행 수: 413396
📌 최종 서비스 업종 개수: 36

📌 최종 서비스 업종 리스트:
['가구', '가전제품', '가전제품수리', '노래방', '문구', '미용실', '부동산중개업', '서적', '세탁소', '슈퍼마켓', '스포츠클럽', '시계및귀금속', '안경', '애완동물', '양식음식점', '여관', '완구', '인테리어', '일반교습학원', '일반의류', '일반의원', '일식음식점', '자동차수리', '자전거 및 기타운송장비', '중식음식점', '철물점', '커피-음료', '컴퓨터및주변장치판매', '패스트푸드점', '편의점', '피부관리실', '한식음식점', '핸드폰', '호프-간이주점', '화장품', '화초']

✅ 저장 완료: outputs/store_final_filtered.csv


In [41]:
import pandas as pd

# ==============================
# 1️⃣ 파일 로드
# ==============================4

def read_csv_kor(path, usecols=None):
    for enc in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            print(f"시도 인코딩: {enc}")
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


# ==============================
# 파일 로드 (안전 버전)
# ==============================

store = read_csv_kor("outputs/store_final_filtered.csv")

sales = read_csv_kor(
    "../data/raw_data/sales_info/서울시(추정매출-행정동).csv"
)

population = read_csv_kor(
    "../data/raw_data/population_info/서울시(유동인구-행정동).csv"
)

print("✅ 파일 로드 완료")

# ==============================
# 2️⃣ 컬럼명 통일
# ==============================

sales = sales.rename(columns={
    "행정동_코드": "행정동코드",
    "서비스_업종_코드_명": "서비스업종"
})

population = population.rename(columns={
    "행정동_코드": "행정동코드"
})


# ==============================
# 3️⃣ store 집계 (공급)
# ==============================

store_group = (
    store
    .groupby(["행정동코드", "서비스업종"])
    .size()
    .reset_index(name="점포수")
)


# ==============================
# 4️⃣ sales 집계 (매출)
# ==============================

sales_group = (
    sales
    .groupby(["행정동코드", "서비스업종"])["당월_매출_금액"]
    .sum()
    .reset_index(name="총매출")
)


# ==============================
# 5️⃣ population 정리 (유동인구)
# ==============================

population_group = population[["행정동코드", "총_유동인구_수"]].copy()


# ==============================
# 6️⃣ 병합 시작
# ==============================

# store + sales
merged = pd.merge(
    store_group,
    sales_group,
    on=["행정동코드", "서비스업종"],
    how="left"
)

# population 추가
merged = pd.merge(
    merged,
    population_group,
    on="행정동코드",
    how="left"
)

# NaN 처리
merged["총매출"] = merged["총매출"].fillna(0)
merged["총_유동인구_수"] = merged["총_유동인구_수"].fillna(0)


# ==============================
# 7️⃣ 핵심 파생변수 생성
# ==============================

# 점포당 매출
merged["점포당매출"] = merged["총매출"] / (merged["점포수"] + 1)

# 유동 대비 매출 효율
merged["유동대비매출"] = merged["총매출"] / (merged["총_유동인구_수"] + 1)


# ==============================
# 8️⃣ 저장
# ==============================

merged.to_csv("outputs/final_merged_data.csv",
              index=False,
              encoding="utf-8-sig")

print("✅ 통합 데이터 생성 완료")
print("행 수:", len(merged))
print(merged.head())


시도 인코딩: utf-8-sig
시도 인코딩: utf-8-sig
시도 인코딩: cp949
시도 인코딩: utf-8-sig
시도 인코딩: cp949
✅ 파일 로드 완료
✅ 통합 데이터 생성 완료
행 수: 376091
      행정동코드 서비스업종  점포수  총매출   총_유동인구_수  점포당매출  유동대비매출
0  11110515  가전제품    2  0.0  3274798.0    0.0     0.0
1  11110515  가전제품    2  0.0  3549035.0    0.0     0.0
2  11110515  가전제품    2  0.0  3322244.0    0.0     0.0
3  11110515  가전제품    2  0.0  3580355.0    0.0     0.0
4  11110515  가전제품    2  0.0  3461402.0    0.0     0.0


In [42]:
import numpy as np

# ==============================
# 1️⃣ 행정동 전체 점포수
# ==============================
merged["행정동_전체점포수"] = (
    merged.groupby("행정동코드")["점포수"]
    .transform("sum")
)

# 업종 포화도 (경쟁 강도)
merged["업종포화도"] = (
    merged["점포수"] / (merged["행정동_전체점포수"] + 1)
)


# ==============================
# 2️⃣ 행정동 전체 매출
# ==============================
merged["행정동_전체매출"] = (
    merged.groupby("행정동코드")["총매출"]
    .transform("sum")
)

# 업종 매출 점유율
merged["매출점유율"] = (
    merged["총매출"] / (merged["행정동_전체매출"] + 1)
)


# ==============================
# 3️⃣ 점포당 매출 재확인
# ==============================
merged["점포당매출"] = (
    merged["총매출"] / (merged["점포수"] + 1)
)


# ==============================
# 4️⃣ 유동 대비 매출
# ==============================
merged["유동대비매출"] = (
    merged["총매출"] / (merged["총_유동인구_수"] + 1)
)


# ==============================
# 5️⃣ 정규화 (0~1 스케일)
# ==============================
def minmax(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

merged["점포당매출_norm"] = minmax(merged["점포당매출"])
merged["매출점유율_norm"] = minmax(merged["매출점유율"])
merged["유동대비매출_norm"] = minmax(merged["유동대비매출"])
merged["업종포화도_norm"] = minmax(merged["업종포화도"])


# ==============================
# 6️⃣ 최종 추천 점수 계산
# ==============================
merged["추천점수"] = (
    0.4 * merged["점포당매출_norm"] +
    0.3 * merged["매출점유율_norm"] +
    0.2 * merged["유동대비매출_norm"] -
    0.1 * merged["업종포화도_norm"]   # 경쟁은 감점
)


# ==============================
# 7️⃣ 결과 확인
# ==============================
print("✅ 파생변수 생성 완료")
print(merged[[
    "행정동코드",
    "서비스업종",
    "점포수",
    "총매출",
    "업종포화도",
    "매출점유율",
    "점포당매출",
    "유동대비매출",
    "추천점수"
]].head())


# ==============================
# 8️⃣ 저장
# ==============================
merged.to_csv(
    "outputs/final_with_features.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ 저장 완료: outputs/final_with_features.csv")


✅ 파생변수 생성 완료
      행정동코드 서비스업종  점포수  총매출     업종포화도  매출점유율  점포당매출  유동대비매출     추천점수
0  11110515  가전제품    2  0.0  0.000102    0.0    0.0     0.0 -0.00003
1  11110515  가전제품    2  0.0  0.000102    0.0    0.0     0.0 -0.00003
2  11110515  가전제품    2  0.0  0.000102    0.0    0.0     0.0 -0.00003
3  11110515  가전제품    2  0.0  0.000102    0.0    0.0     0.0 -0.00003
4  11110515  가전제품    2  0.0  0.000102    0.0    0.0     0.0 -0.00003
✅ 저장 완료: outputs/final_with_features.csv


In [47]:
merged[merged["행정동코드"] == "11110540"] \
    .sort_values("추천점수", ascending=False) \
    .head(5)


,행정동코드,서비스업종,점포수,총매출,총_유동인구_수,점포당매출,유동대비매출,행정동_전체점포수,업종포화도,행정동_전체매출,매출점유율,점포당매출_norm,매출점유율_norm,유동대비매출_norm,업종포화도_norm,추천점수
